# Notebook 2: AI Alpha Generation (PyTorch Deep Learning & QuantLoss)

Trong notebook này, chúng ta sẽ huấn luyện mạng nơ-ron sâu (`AlphaMLP` với BatchNorm, Dropout, GELU) trên tập dữ liệu chuỗi thời gian chứng khoán Việt Nam:
1. Tách tập dữ liệu theo thời gian (Train, Validation, Test) để tránh Look-ahead Bias.
2. Định nghĩa hàm mất mát `QuantLoss` tối ưu cho Quant: kết hợp giữa **MSE Loss** và **Information Coefficient (IC) Loss**.
3. Huấn luyện mô hình và theo dõi sự hội tụ.
4. Đánh giá chỉ số IC Out-of-Sample và dự đoán vector $\mu$.


In [8]:
!pwd
!ls -al


/content
total 16
drwxr-xr-x 1 root root 4096 Jun  4 13:32 .
drwxr-xr-x 1 root root 4096 Jul 28 03:17 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:32 .config
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import Config
from src.data_pipeline import DataFetcher, DataCleaner, FeatureEngineer, DataPreprocessor
from src.models import create_dataloaders, AlphaMLP, AlphaTrainer, AlphaPredictor

%matplotlib inline
sns.set_theme(style='whitegrid')


ModuleNotFoundError: No module named 'src'

## 1. Chuẩn bị tập dữ liệu huấn luyện (Chronological Split)


In [ ]:
fetcher = DataFetcher()
raw_data = fetcher.fetch_all()
cleaned_data = DataCleaner().clean_and_align(raw_data)
feature_data = FeatureEngineer().compute_all_features(cleaned_data)
preprocessor = DataPreprocessor()
normalized_data = preprocessor.normalize_features(feature_data)

master_df = preprocessor.prepare_tabular_dataset(normalized_data, Config.START_DATE, Config.END_DATE)
train_loader, val_loader, test_loader, test_df = create_dataloaders(master_df, batch_size=64)
print(f'Đã khởi tạo DataLoaders cho PyTorch. Kích thước tập Test: {len(test_df)} mẫu.')


## 2. Khởi tạo & Huấn luyện mạng AlphaMLP với QuantLoss (MSE + IC)


In [ ]:
model = AlphaMLP(input_dim=len(FeatureEngineer.get_feature_names()))
trainer = AlphaTrainer(model, lr=1e-3)

# Huấn luyện mô hình
history = trainer.fit(train_loader, val_loader, epochs=20)


## 3. Trực quan hóa đường cong mất mát (Loss Curve) & Information Coefficient (IC)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss', color='blue')
axes[0].plot(history['val_loss'], label='Val Loss', color='red')
axes[0].set_title('Đường cong Mất mát (MSE + IC Loss)')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history['val_ic'], label='Validation Rank IC', color='green', marker='o')
axes[1].axhline(0.0, color='gray', linestyle='--')
axes[1].set_title('Độ tương quan thứ hạng (Information Coefficient - IC)')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Pearson IC')
axes[1].legend()
plt.tight_layout()
plt.show()


## 4. Kiểm chứng suy luận Out-of-Sample (Predicting $\mu$)


In [ ]:
predictor = AlphaPredictor(model)
mu_preds = predictor.predict_all(test_df)
print('Kết quả dự báo lợi suất kỳ vọng mu trên tập Test:')
mu_preds[['date', 'symbol', 'close', 'target_ret_1d', 'predicted_mu']].tail(10)
